# 02 - First Factor: 12-1 Month Momentum

The first classical alpha factor. **12-1 momentum** is the cumulative return from
~12 months ago up to ~1 month ago: lookback of 252 trading days, skipping the
most recent 21. The recent month is dropped on purpose - very short-horizon
returns mean-revert and would contaminate a momentum signal.

This notebook does two things:

1. Computes the cross-sectional momentum factor on a single date.
2. Evaluates it over time with the **information coefficient (IC)** - the daily
   rank correlation between the factor and the forward 21-day return.

**Prerequisite:** prices must be ingested (`make data`).

**Price convention.** `loader.close` returns raw, split/dividend-adjusted price
*levels*. `momentum_12_1` defaults to `log_prices=False`, meaning it treats its
input as raw levels and takes logs internally - so passing `loader.close`
straight in yields a proper log-return momentum signal.

In [1]:
from qer.data.loader import DataLoader
from qer.factors.momentum import momentum_12_1

In [2]:
loader = DataLoader()

## Compute the factor on one date

We pick a single as-of date and compute one momentum value per ticker. The
function uses only data through the as-of date (no look-ahead) and returns `NaN`
- never a misleading zero - for tickers without enough history.

In [3]:
prices_df = loader.close
as_of_date = "2025-08-08"

In [4]:
# momentum_12_1 is the canonical 252-day lookback / 21-day skip variant, so it
# takes just the price matrix and the as-of date. (For custom windows, call
# qer.factors.momentum.momentum directly with lookback_days / skip_days.)
computed_momentum = momentum_12_1(prices_df, as_of_date)

In [5]:
computed_momentum

A      -0.085250
AA      0.083851
AAL     0.329234
AAP     0.043441
AAPL    0.029355
          ...   
YUM     0.126714
ZBH    -0.131910
ZBRA    0.042640
ZION    0.232629
ZTS    -0.145522
Name: mom_252d_21d_2025-08-08, Length: 848, dtype: float64

The values are log returns over the 11-month window: mostly in roughly the
-0.5 to +0.5 range, centred near zero, with the cross-sectional spread being
exactly the dispersion the factor will rank on. Large-magnitude tails are real
(big winners/losers), not a unit error - if these came back in the tens, that
would signal raw price *differences* instead of returns.

In [6]:
computed_momentum.describe()

count    636.000000
mean       0.103213
std        0.363599
min       -4.127134
25%       -0.046628
50%        0.111191
75%        0.269172
max        1.865980
Name: mom_252d_21d_2025-08-08, dtype: float64

## Evaluate the factor: information coefficient

A single date tells us nothing about whether the signal *works*. The IC harness
in `qer.diagnostics.factor_ic` computes, for every trading day in the window, the
Spearman rank correlation between the momentum factor and the **forward** 21-day
return across the active universe, then summarises and plots it.

Reading the headline numbers (roadmap calibration for a known classical factor):

- **Mean IC** ~ 0.01-0.04 daily is a normal, real-but-noisy signal.
- **IC IR (annualised)** ~ 0.3-1.0 measures *stability*, not significance. It is a
  descriptive mean/std x sqrt(252) ratio and is deliberately **not** Newey-West
  adjusted - there is no standard error in it to correct.
- **t-stat.** The cell prints two. The *naive* one (mean/std x sqrt(N)) is inflated
  here, because daily ICs against a 21-day forward return overlap by 20 of 21 days.
  The **Newey-West** t-stat (lag = h-1 = 20) corrects that autocorrelation and is the
  one to trust; ~1.5-3.0 is a good single factor. This is the introductory view - the
  full multiple-testing treatment (BH, deflated Sharpe) lives in notebook 04.

In [7]:
from qer.diagnostics.factor_ic import run_momentum_ic_analysis

In [13]:
ic, summary = run_momentum_ic_analysis(loader, years=15)

Computing IC over 3752 trading days (2010-12-30 to 2025-11-28)...

Momentum 12-1 vs 21-day forward return
  N days computed:    3752
  Mean IC:            +0.0074
  Std IC:             0.2068
  IC IR (annualized): +0.566
  t-stat of IC:       +2.18
  Hit rate:           54.5%

Chart saved to /home/alex/Desktop/quant-equity-research/data/audit/momentum_ic.png


In [14]:
ic.describe()

count    3752.000000
mean        0.007370
std         0.206774
min        -0.767052
25%        -0.118176
50%         0.020014
75%         0.146222
max         0.767872
Name: ic, dtype: float64

## Takeaways

A small positive mean IC with a daily std of ~0.15-0.20 is exactly the texture of
a real single factor: individually noisy day to day, but with a persistent
positive tilt that an annualised IC IR around 0.5+ captures. The rolling-mean
line in the chart shows the signal weakening or inverting during momentum-crash
regimes (e.g. sharp reversals after drawdowns), which is expected and worth
remembering when we later combine factors and manage risk.

Next: add the remaining classical factors behind the same interface and the same
IC harness, so every factor is judged on a like-for-like basis.